In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# A Theorem Prover for First-Order Logic without Equality

This notebook implements a resolution-based theorem prover in OCaml.

We need the parser for first order formulas.  In our parser, variables start with an uppercase letter, and predicate and function symbols start with a lowercase letter.

The resolution calculus works with clauses.  The notebook `01-FOL-CNF.ipynb` implements the function `normalize` that turns a formula into a set of clauses.  It imports the notebook `FOL-Parser.ipynb`, which provides the function `parse`.

In [ ]:
import_notebook "01-FOL-CNF.ipynb"

The notebook `02-Unification.ipynb` implements [unification](https://en.wikipedia.org/wiki/Unification_(computer_science)) via the algorithm of [Martelli and Montanari](https://dl.acm.org/doi/pdf/10.1145/357162.357169).  It provides the function `unify`.

In [ ]:
import_notebook "02-Unification.ipynb"

## Domain Types

The notebook `01-FOL-CNF.ipynb` defines the types of literals and clauses:
```
type literal =
  | Pos of string * term list   (* p(t1, ..., tn) *)
  | Neg of string * term list   (* ¬p(t1, ..., tn) *)

type clause = literal RSet.t
type cnf    = clause RSet.t
```
It also provides the functions `clause_to_string` and `cnf_to_string`, which convert clauses and sets of clauses into strings.

In [ ]:
let clauses = normalize (parse "∀G:∀C:(grandparent(G, C) ↔ ∃P:(parent(G, P) ∧ parent(P, C)))")

In [ ]:
print_endline (cnf_to_string clauses)

In TypeScript, literals are nested tuples.  The TypeScript version of this notebook therefore needs a number of *type guards*, i.e. functions that check at run time whether an object is a term, an atomic formula, or a negated atomic formula, and functions that safely take these objects apart.  In OCaml, a literal is taken apart by pattern matching, and the compiler checks that all cases are covered.  We only need two small functions.

The function `is_negative` takes one argument:
- `l` is a literal.

It returns `true` if `l` is a negated atomic formula.

The function `atom_term` takes one argument:
- `l` is a literal.

It returns the atomic formula of `l` as a term: Both $p(t_1, \cdots, t_n)$ and $\neg p(t_1, \cdots, t_n)$ are mapped to the term $p(t_1, \cdots, t_n)$.  We need this function since the function `unify` from the notebook `02-Unification.ipynb` unifies terms.  Syntactically, an atomic formula looks like a term whose function symbol is the predicate symbol.

In [ ]:
let is_negative (l : literal) : bool =
  match l with
  | Pos _ -> false
  | Neg _ -> true

let atom_term (l : literal) : term =
  match l with
  | Pos (p, ts) | Neg (p, ts) -> Fun (p, ts)

In [ ]:
let l = Neg ("parent", [Var "G"; Fun ("sk1", [Var "G"; Var "C"])]) in
(is_negative l, term_to_string (atom_term l))

## Variable Extraction and Substitution

The function `literal_variables` takes one argument:
- `l` is a literal.

It returns the set of all variables occurring in `l`.  The function `term_variables` is defined in the notebook `01-FOL-CNF.ipynb`.

In [ ]:
let literal_variables (l : literal) : string RSet.t =
  match l with
  | Pos (_, ts) | Neg (_, ts) -> RSet.unions (List.map term_variables ts)

The function `clause_variables` takes one argument:
- `c` is a clause.

It returns the set of all variables occurring in `c`.

In [ ]:
let clause_variables (c : clause) : string RSet.t =
  RSet.flat_map literal_variables c

In [ ]:
RSet.iter (fun c -> Printf.printf "clause_variables %s = %s\n" (clause_to_string c) (RSet.to_string Fun.id (clause_variables c))) clauses

The function `apply_clause` takes two arguments:
- `sigma` is a substitution,
- `c` is a clause.

It applies `sigma` to all literals of `c`.  The function `apply_literal` is defined in the notebook `01-FOL-CNF.ipynb`.  The TypeScript version of this notebook needs separate functions for renamings, which map variables to variables, and for unifiers, which map variables to terms.  In OCaml, both are substitutions.  As `RSet.map` removes duplicates, literals that become equal when `sigma` is applied are merged.

In [ ]:
let apply_clause (sigma : substitution) (c : clause) : clause =
  RSet.map (apply_literal sigma) c

The function `rename_variables` takes two arguments:
- `f` is a clause,
- `g` is a clause.

It renames the variables in the clause `f` so that they are different from the variables occurring in `g`.  The new variables are upper case letters from the list `ascii_uppercase`, which is defined in the notebook `01-FOL-CNF.ipynb`.

**Implementation:** `old_vars` is the list of the variables of `f`, and `fresh_vars` is the list of the letters that do not occur in `g`.  `List.take n l` returns the first `n` elements of the list `l`, and `List.combine` pairs the variables of `f` with the new variables.  If there are not enough letters, an exception is raised.

In [ ]:
let rename_variables (f : clause) (g : clause) : clause =
  let old_vars   = RSet.to_list (clause_variables f) in
  let g_vars     = clause_variables g in
  let fresh_vars = List.filter (fun v -> not (RSet.mem v g_vars)) ascii_uppercase in
  let n          = List.length old_vars in
  if List.length fresh_vars < n then failwith "rename_variables: not enough new variables";
  let sigma = RMap.of_list (List.combine old_vars (List.map (fun v -> Var v) (List.take n fresh_vars))) in
  apply_clause sigma f

In [ ]:
RSet.iter (fun c -> print_endline (clause_to_string (rename_variables c c))) clauses

## A Calculus for First Order Logic

The resolution rule is an inference rule that is defined as follows: If
 * $C_1$ and $C_2$ are clauses from first order logic,
 * $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
 * the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$ \mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then
$$\frac{C_1 \cup\{ p(s_1,\cdots,s_n)\} \quad\quad \{\neg p(t_1,\cdots,t_n)\} \cup C_2}{
                 C_1\mu \cup C_2\mu}
$$
is an application of the resolution rule.

The function `resolve` takes two arguments:
- `c1` is a clause,
- `c2` is a clause.

It returns the set of all clauses that can be inferred from `c1` and `c2` by applying the resolution rule.  Before the rule is applied, the variables of `c2` are renamed, so that `c1` and `c2` have no variables in common.

**Implementation:** For every literal `l1` of `c1`, `RSet.filter_map` computes the set of resolvents of `l1` with the literals `l2` of `c2`.  A pair of literals is resolved if one of them is negated and the other one is not, and if their atomic formulas are unifiable.  `RSet.flat_map` computes the union of these sets.

In [ ]:
let resolve (c1 : clause) (c2 : clause) : clause RSet.t =
  let c2 = rename_variables c2 c1 in
  RSet.flat_map
    (fun l1 ->
       RSet.filter_map
         (fun l2 ->
            if is_negative l1 = is_negative l2 then
              None
            else
              match unify (atom_term l1) (atom_term l2) with
              | Some mu -> Some (apply_clause mu (RSet.union (RSet.remove l1 c1) (RSet.remove l2 c2)))
              | None    -> None)
         c2)
    c1

In the following example, the variable `X` of the second clause is renamed to `A` before the resolution rule is applied.

In [ ]:
let c1 : clause = RSet.of_list [Pos ("p", [Var "X"]); Pos ("q", [Var "X"])]
let c2 : clause = RSet.of_list [Neg ("p", [Fun ("f", [Var "X"])])]

In [ ]:
RSet.to_string clause_to_string (resolve c1 c2)

## Factorization

A calculus which only contains the resolution rule is not complete.  We also need the <em style="color:blue;">factorization</em> rule.  If
1. $C$ is a clause from first order logic,
2. $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
3. the syntactical equation $p(s_1,\cdots,s_n)  \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$\mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$

then both
$\displaystyle \frac{C \cup \bigl\{p(s_1,\cdots,s_n),\, p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{p(s_1,\cdots,s_n)\mu\bigr\} } $
and
$\displaystyle \frac{C \cup \bigl\{ \neg p(s_1,\cdots,s_n),\, \neg p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{\neg p(s_1,\cdots,s_n)\mu\bigr\} }
$
are applications of the factorization rule.

The function `factorize` takes one argument:
- `c` is a clause from first order logic.

It returns the set of all clauses that can be derived from `c` via factorization.

**Implementation:** The local function `pairs` takes a list `ls` and returns the list of all pairs `(l1, l2)` such that `l1` occurs before `l2` in `ls`.  Hence every pair of different literals of `c` is considered once.  If both literals are positive or both are negative and their atomic formulas are unifiable with the most general unifier `mu`, the clause $c\mu$ is a factor of `c`.

In [ ]:
let factorize (c : clause) : clause RSet.t =
  let rec pairs ls =
    match ls with
    | []        -> []
    | l :: rest -> List.map (fun l2 -> (l, l2)) rest @ pairs rest
  in
  RSet.of_list
    (List.filter_map
       (fun (l1, l2) ->
          if is_negative l1 <> is_negative l2 then
            None
          else
            match unify (atom_term l1) (atom_term l2) with
            | Some mu -> Some (apply_clause mu c)
            | None    -> None)
       (pairs (RSet.to_list c)))

The formulas
$$ \forall X: \forall Y: p(f(X),Y) \vee \forall U: \forall V: p(U, g(V)) $$
and
$$ \forall X: \forall Y: \neg p(f(X),Y) \vee \forall U: \forall V: \neg p(U, g(V)) $$
are inconsistent.  However, the resolution rule alone is not sufficient to show this, since every resolvent of the two clauses contains a literal.  The function `RSet.choose` returns the smallest element of a set.  Here, the sets contain only one clause.

In [ ]:
let d1 = RSet.choose (normalize (parse "∀X:∀Y:p(f(X),Y) ∨ ∀U:∀V:p(U,g(V))"))
let d2 = RSet.choose (normalize (parse "∀X:∀Y:¬p(f(X),Y) ∨ ∀U:∀V:¬p(U,g(V))"))

In [ ]:
(clause_to_string d1, clause_to_string d2)

In [ ]:
RSet.to_string clause_to_string (resolve d1 d2)

Factorization reduces both clauses to a single literal.  Then the empty clause can be derived by resolution.

In [ ]:
let d3 = RSet.choose (factorize d1)
let d4 = RSet.choose (factorize d2)

In [ ]:
(clause_to_string d3, clause_to_string d4)

In [ ]:
RSet.to_string clause_to_string (resolve d3 d4)

## Automatic Theorem Proving

In order to be able to construct a proof once the empty clause has been derived, we have to remember how every clause has been derived.  A clause is derived either by resolution from two clauses or by factorization from one clause.  This is expressed by the type `reason`.  In TypeScript, a reason is a tuple containing either one or two clauses.

In [ ]:
type reason =
  | Resolution    of clause * clause
  | Factorization of clause

The function `infere` takes one argument:
- `clauses` is a set of clauses.

It returns the list of all pairs `(c, r)` such that the clause `c` results from
<ol>
    <li>the resolution of two clauses $c_1, c_2 \in \texttt{clauses}$, or</li>
    <li>the factorization of a clause $c_1 \in \texttt{clauses}$.</li>
</ol>
The reason `r` records how `c` has been derived.  A clause can be derived in several ways, so it can occur in several pairs.

In [ ]:
let infere (clauses : cnf) : (clause * reason) list =
  let cs = RSet.to_list clauses in
  List.concat_map
    (fun c1 ->
       List.concat_map
         (fun c2 -> List.map (fun c -> (c, Resolution (c1, c2))) (RSet.to_list (resolve c1 c2)))
         cs
       @ List.map (fun c -> (c, Factorization c1)) (RSet.to_list (factorize c1)))
    cs

The function `saturate` takes one argument:
- `cs` is a set of clauses.

It tries to infer the empty clause from `cs`.  If the empty clause is derived, it returns `Some reasons`, where `reasons` is a map that assigns to every derived clause the reason how this clause has been derived.  If no new clauses can be derived, but the empty clause has not been derived, then `cs` is satisfiable and `None` is returned.  Usually, however, there are infinitely many clauses that can be derived from a satisfiable set of clauses.  Then the function runs until the memory is exhausted.

**Implementation:** The local function `loop` takes the set `clauses` of the clauses derived so far, the map `reasons`, and a counter `cnt` of the iterations.  In every iteration, all clauses that can be inferred from `clauses` are computed.
- `derived` is the list of the inferred clauses that are new, together with their reasons.
- `new_reasons` maps every new clause to the reason of its first occurrence in `derived`.  As `RMap.of_list` keeps the last pair of every key, the list `derived` is reversed first.
- The keys of `new_reasons` are added to `clauses`.  As the clauses in `new_reasons` and `reasons` are different, `RMap.of_list` can combine the two maps into one.

Adding the new clauses all at once is much faster than adding them one by one, since `RSet.add` and `RMap.add` take time linear in the size of the set or map.

In [ ]:
let saturate (cs : cnf) : (clause, reason) RMap.t option =
  let rec loop (clauses : cnf) (reasons : (clause, reason) RMap.t) (cnt : int) =
    if RSet.mem RSet.empty clauses then
      Some reasons
    else begin
      let derived     = List.filter (fun (c, _) -> not (RSet.mem c clauses)) (infere clauses) in
      let new_reasons = RMap.of_list (List.rev derived) in
      let clauses     = RSet.union clauses (RMap.keys new_reasons) in
      Printf.printf "cnt = %d, number of clauses: %d\n" cnt (RSet.cardinal clauses);
      if RMap.is_empty new_reasons then
        None
      else
        loop clauses (RMap.of_list (RMap.to_list reasons @ RMap.to_list new_reasons)) (cnt + 1)
    end
  in
  loop cs RMap.empty 1

## Proof Reconstruction

The function `update_proof` takes two arguments:
- `p1` is a proof, i.e. a list of strings describing the steps of the proof,
- `p2` is a proof.

It appends those lines of `p2` to `p1` that do not already occur in `p1`.

In [ ]:
let update_proof (p1 : string list) (p2 : string list) : string list =
  p1 @ List.filter (fun line -> not (List.mem line p1)) p2

The function `construct_proof` takes two arguments:
- `c` is a clause that has been derived,
- `reasons` is a map that records how the clauses have been derived.

It returns a proof of `c` as a list of strings, where every string describes one step of the proof.  Clauses that do not have a reason are axioms.  The proofs of the premises of a step precede the step itself.

In [ ]:
let rec construct_proof (c : clause) (reasons : (clause, reason) RMap.t) : string list =
  match RMap.find_opt c reasons with
  | None ->
      [Printf.sprintf "Axiom:       %s" (clause_to_string c)]
  | Some (Factorization c1) ->
      construct_proof c1 reasons
      @ [Printf.sprintf "Factorization: %s\n⊢            %s" (clause_to_string c1) (clause_to_string c)]
  | Some (Resolution (c1, c2)) ->
      let proof1 = construct_proof c1 reasons in
      let proof2 = construct_proof c2 reasons in
      update_proof proof1 proof2
      @ [Printf.sprintf "Resolution:  %s,\n             %s\n⊢            %s"
           (clause_to_string c1) (clause_to_string c2) (clause_to_string c)]

## Testing with the Red Dragons

According to <a href="https://de.wikipedia.org/wiki/Uwe_Schöning">Uwe Schöning</a>, the theory of red dragons is given by the following axioms:

1. Every dragon is happy if all its children can fly.
2. All red dragons can fly.
3. The children of red dragons are themselves red.

We will show that these axioms imply that all red dragons are happy.  To this end, the formula stating that all red dragons are happy is negated.  Then we will show that the set consisting of the negated formula together with the axioms is inconsistent.

In [ ]:
let s1 = "∀X:(∀Y:(child(Y, X) → canFly(Y)) → happy(X))"
let s2 = "∀X:(red(X) → canFly(X))"
let s3 = "∀X:(red(X) → ∀Y:(child(Y, X) → red(Y)))"
let s4 = "¬∀X:(red(X) → happy(X))"

The formulas are parsed and transformed into clauses.  `RSet.unions` computes the union of the sets of clauses.

In [ ]:
let dragon_clauses = RSet.unions (List.map (fun s -> normalize (parse s)) [s1; s2; s3; s4])

In [ ]:
print_endline (cnf_to_string dragon_clauses)

The map of reasons contains thousands of clauses.  We do not bind it to a top level variable, since the toplevel would print a part of it.  Instead, the proof of the empty clause is printed right away.

In [ ]:
match time "saturate" (fun () -> saturate dragon_clauses) with
| Some reasons -> List.iter print_endline (construct_proof RSet.empty reasons)
| None         -> print_endline "The empty clause can not be derived."